# Embedding model evaluation -- incident similarity retrieval

Decides which multilingual embedding model backs the Knowledge Base module's similarity engine
(`app/modules/knowledge_base/`), by retrieval quality **and** practical cost on the constrained
target hardware (Intel i5-6300U, 16GB RAM, no GPU), not benchmarks alone.

**Candidates** (all served locally through Ollama, verified against the live Ollama library and
this machine's installed `ollama` package before use):

| Model | Disk size | Params | Context |
|---|---|---|---|
| `qwen3-embedding:0.6b` | 639 MB | ~596M | 32K tokens |
| `embeddinggemma:300m` | 622 MB | ~308M | 2K tokens |
| `bge-m3` | 1.16 GB | ~567M | 8K tokens |

**Methodology at a glance** -- pooled retrieval evaluation, same queries and same human judgments
reused across all three models:

1. Sample ~100 query tickets, stratified by each application's share of the corpus.
2. Retrieve each model's top 10 nearest historical tickets (description-only embeddings).
3. Pool the three models' candidates per query into one deduplicated, blinded judgment queue.
4. Judge each (query, candidate) pair once -- binary relevant / not relevant -- reused by every model.
5. Score Precision@10, pool-relative Recall@10, and MRR per model; benchmark latency/memory/size
   separately. No composite score -- quality and cost are reported side by side.

**Decisions locked in before implementation** (confirmed in conversation, not assumed):

- Sampling: stratified by corpus share (not equal-per-app) -- AERO (54 tickets total) gets
  proportionally few queries; this evaluation will have low statistical confidence for AERO
  specifically, and that limitation is called out again in the final section.
- Minimum description length: **none** -- every ticket with a non-empty description is eligible,
  including FCI's very short (2-9 character) entries. This is a deliberate stress test of how each
  model handles low-signal text, not an oversight.
- Near-duplicates: queries are deduplicated by exact description text (no two queries share
  identical text), but duplicate tickets remain valid, retrievable candidates -- VIO alone is 57%
  exact-duplicate descriptions.
- Relevance judgments: binary (relevant / not relevant), not graded.

**Explicitly out of scope here** (see `app/modules/knowledge_base/CLAUDE.md` §11): similarity
threshold/cap tuning, pgvector index type/distance metric, backfill/rebuild, and any input
representation beyond `description` alone -- those are separate, later steps.

**How to run this notebook -- "Run All" will NOT pause for you, read this first:**

`ipywidgets` buttons don't block cell execution, so a single "Run All" runs straight through the
judgment widget with zero judgments recorded and produces meaningless (empty) metrics at the end.
The actual workflow has three steps:

1. Run every cell top to bottom **up through the "judgment-store" cell** (the cell just above the
   judgment widget) -- this samples/freezes the benchmark, generates and caches embeddings for all
   three models (slow, CPU-only -- expect ~30 minutes the first time, instant on reruns since it's
   cached), runs retrieval, and loads `judgments` fresh from disk.
2. Run the widget cell below it and judge at your own pace. Safe to stop anytime -- each click
   appends to `cache/judgments.jsonl` immediately, resumable across sessions.
3. When ready to see results (even partial), **re-run starting from the "judgment-store" cell**
   (Jupyter: "Run Selected Cell and All Below" on that cell) and continue down through the end.
   Re-running from section 7 alone is NOT enough -- `judgments` is only loaded once, in the
   judgment-store cell, and clicking the widget's buttons never updates that in-memory variable,
   only the file on disk.

All experiment artifacts live under `cache/` (git-ignored, same as `data/` and `storage/` --
regenerable, and may echo raw ticket content).


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, "../..")

from app.scripts.seeding.ticket_data.processed_tickets import load_processed_tickets

RANDOM_SEED = 42
TOP_K = 10
N_QUERIES = 100
LATENCY_SAMPLE_SIZE = 30

MODEL_TAGS = ["qwen3-embedding:0.6b", "embeddinggemma:300m", "bge-m3"]
OLLAMA_HOST = "http://localhost:11434"

CACHE_DIR = Path("cache")
EMBEDDINGS_DIR = CACHE_DIR / "embeddings"
RETRIEVAL_DIR = CACHE_DIR / "retrieval"
BENCHMARK_CORPUS_PATH = CACHE_DIR / "benchmark_corpus.csv"
BENCHMARK_QUERIES_PATH = CACHE_DIR / "benchmark_queries.json"
JUDGMENTS_PATH = CACHE_DIR / "judgments.jsonl"
MODEL_INFO_PATH = CACHE_DIR / "model_info.json"
RESOURCE_BENCHMARK_PATH = CACHE_DIR / "resource_benchmark.json"

for directory in (CACHE_DIR, EMBEDDINGS_DIR, RETRIEVAL_DIR):
	directory.mkdir(parents=True, exist_ok=True)


## 1. Load the ticket corpus

Reuses `app.scripts.seeding.ticket_data.processed_tickets.load_processed_tickets()` -- the same historical-data loader the ticket seeder itself uses -- rather than re-parsing `data/processed/*.json`.

In [ ]:
def build_corpus() -> pd.DataFrame:
	"""One row per historical ticket. Position in this fixed concatenation (FCI, COLORIS, AERO,
	VIO, in load_processed_tickets()'s order) is its stable id for this experiment -- the source
	files are a static historical export, not a live table, so this id never changes across reruns.
	"""
	rows = load_processed_tickets()
	records = [
		{
			"id": i,
			"application": row.application.value,
			"category": row.category.value,
			"title": row.title,
			"description": row.description,
			"resolution_notes": row.resolution_notes,
		}
		for i, row in enumerate(rows)
	]
	return pd.DataFrame.from_records(records)


def has_meaningful_description(text: str | None) -> bool:
	return text is not None and text.strip() != ""


corpus_raw = build_corpus()
corpus = corpus_raw[corpus_raw["description"].apply(has_meaningful_description)].reset_index(drop=True)

print(f"Loaded {len(corpus_raw)} historical tickets, {len(corpus)} eligible (non-empty description).")
print(pd.concat([
	corpus_raw.groupby("application").size().rename("total"),
	corpus.groupby("application").size().rename("eligible"),
], axis=1))


## 2. Establish the shared evaluation set (frozen benchmark)

Sampled once, then frozen to `cache/benchmark_corpus.csv` / `cache/benchmark_queries.json`. Every later run loads the frozen set instead of resampling, so embeddings, retrieval results, and judgments all stay pinned to the exact same tickets even if `data/processed/*.json` changes later.

In [ ]:
def sample_queries(eligible: pd.DataFrame, n_queries: int, seed: int) -> pd.DataFrame:
	"""Stratified by each application's share of the eligible corpus (not equal-per-app) --
	confirmed: AERO (54 tickets) gets proportionally few queries rather than being padded to match
	the larger applications. Deduplicated by exact description text within each stratum so none of
	the n_queries slots are wasted judging the same text twice -- VIO alone is 57% exact-duplicate
	descriptions. Duplicate tickets are NOT removed from the retrievable corpus, only from the
	*query* pool, so recurring incidents remain fully findable as candidates.
	"""
	rng = np.random.default_rng(seed)
	app_counts = eligible.groupby("application").size()
	raw_shares = app_counts / app_counts.sum() * n_queries
	shares = raw_shares.round().astype(int)

	diff = n_queries - shares.sum()
	if diff != 0:
		remainders = raw_shares - shares
		order = remainders.sort_values(ascending=diff < 0).index
		for app in order[: abs(diff)]:
			shares[app] += 1 if diff > 0 else -1

	selected_ids: list[int] = []
	for app, quota in shares.items():
		stratum = eligible[eligible["application"] == app]
		deduped = stratum.drop_duplicates(subset="description", keep="first")
		quota = min(quota, len(deduped))
		chosen = rng.choice(deduped["id"].to_numpy(), size=quota, replace=False)
		selected_ids.extend(int(x) for x in chosen)

	return eligible[eligible["id"].isin(selected_ids)].reset_index(drop=True)


In [ ]:
if BENCHMARK_CORPUS_PATH.exists() and BENCHMARK_QUERIES_PATH.exists():
	print("Frozen benchmark found on disk -- loading it (not resampling), so results stay comparable across reruns.")
	corpus = pd.read_csv(BENCHMARK_CORPUS_PATH, encoding="utf-8")
	query_ids = json.loads(BENCHMARK_QUERIES_PATH.read_text(encoding="utf-8"))["query_ids"]
else:
	print("No frozen benchmark yet -- sampling now. This happens ONCE; delete cache/benchmark_*.* to resample.")
	queries_df = sample_queries(corpus, N_QUERIES, RANDOM_SEED)
	query_ids = queries_df["id"].tolist()
	corpus.to_csv(BENCHMARK_CORPUS_PATH, index=False, encoding="utf-8")
	BENCHMARK_QUERIES_PATH.write_text(
		json.dumps({"query_ids": query_ids, "random_seed": RANDOM_SEED, "n_queries": N_QUERIES}, indent=2),
		encoding="utf-8",
	)

corpus_by_id = corpus.set_index("id")
queries_df = corpus_by_id.loc[query_ids].reset_index()

print(f"{len(corpus)} candidates in the retrievable corpus, {len(query_ids)} frozen queries.")
print(pd.concat([
	corpus.groupby("application").size().rename("candidates"),
	queries_df.groupby("application").size().rename("queries"),
], axis=1))


## 3. Ollama setup & model bookkeeping

Records each model's digest, disk size, architecture, and embedding dimension once, so the final comparison is traceable to an exact model build, not just a name.

In [ ]:
import ollama

_client = ollama.Client(host=OLLAMA_HOST)


class EmbeddingFailure(RuntimeError):
	"""Raised when Ollama fails to produce an embedding after all retries."""


def _full_tag(model_tag: str) -> str:
	"""`list()`/`ps()` always report a fully-qualified tag (e.g. `bge-m3` -> `bge-m3:latest`) even
	when the model was referenced without one, so lookups against those responses must normalize
	first or an untagged reference like `bge-m3` never matches.
	"""
	return model_tag if ":" in model_tag else f"{model_tag}:latest"


def embed_one(model_tag: str, text: str, *, max_attempts: int = 3, base_delay: float = 1.0) -> tuple[list[float], int]:
	"""Returns (embedding, total_duration_ns) -- the duration is Ollama's own server-side timing,
	used later for latency benchmarking instead of Python-side wall-clock. Retries transient
	failures (model still loading, connection hiccups) with exponential backoff.
	"""
	last_error: Exception | None = None
	for attempt in range(1, max_attempts + 1):
		try:
			response = _client.embed(model=model_tag, input=text)
			return list(response.embeddings[0]), response.total_duration
		except Exception as exc:
			last_error = exc
			if attempt < max_attempts:
				time.sleep(base_delay * (2 ** (attempt - 1)))
	raise EmbeddingFailure(f"Failed to embed with {model_tag!r} after {max_attempts} attempts: {last_error}") from last_error


def get_model_info(model_tag: str) -> dict:
	"""Static bookkeeping captured once per model: digest + disk size from `list`, architecture and
	embedding dimension from `show`.
	"""
	list_entry = next(m for m in _client.list().models if m.model == _full_tag(model_tag))
	show = _client.show(model_tag)
	embedding_dimension = next(
		(v for k, v in show.modelinfo.items() if k.endswith("embedding_length")), None
	)
	max_context_length = next(
		(v for k, v in show.modelinfo.items() if k.endswith("context_length")), None
	)
	return {
		"model_tag": model_tag,
		"digest": list_entry.digest[:19],
		"disk_size_bytes": list_entry.size,
		"family": show.details.family,
		"parameter_size": show.details.parameter_size,
		"quantization_level": show.details.quantization_level,
		"embedding_dimension": embedding_dimension,
		"max_context_length": max_context_length,
		"capabilities": show.capabilities,
	}


In [ ]:
model_info: dict = json.loads(MODEL_INFO_PATH.read_text(encoding="utf-8")) if MODEL_INFO_PATH.exists() else {}

for tag in MODEL_TAGS:
	if tag in model_info:
		continue
	model_info[tag] = get_model_info(tag)
	MODEL_INFO_PATH.write_text(json.dumps(model_info, indent=2), encoding="utf-8")  # checkpoint after each model

pd.DataFrame(model_info).T


## 4. Embedding generation (cached, resumable)

Embeds every eligible ticket's `description` (never title, metadata, or resolution notes -- fixed scope) with each candidate model. Per-item caching means an interrupted run picks up where it left off instead of re-embedding from scratch.

In [ ]:
def _npz_path(model_tag: str) -> Path:
	return EMBEDDINGS_DIR / f"{model_tag.replace(':', '_')}.npz"


def load_cached_embeddings(model_tag: str) -> dict[int, np.ndarray]:
	path = _npz_path(model_tag)
	if not path.exists():
		return {}
	data = np.load(path)
	return {int(i): data["embeddings"][row] for row, i in enumerate(data["ids"])}


def save_cached_embeddings(model_tag: str, embeddings: dict[int, np.ndarray]) -> None:
	ids = np.array(sorted(embeddings))
	matrix = np.stack([embeddings[i] for i in ids]).astype(np.float32)
	np.savez_compressed(_npz_path(model_tag), ids=ids, embeddings=matrix)


def embed_corpus(model_tag: str, corpus_df: pd.DataFrame, *, checkpoint_every: int = 25) -> dict[int, np.ndarray]:
	"""Embeds every eligible ticket's description with `model_tag`, skipping ids already cached
	from a previous (possibly interrupted) run, and checkpointing periodically so a crash never
	loses more than `checkpoint_every` items of CPU-bound work.
	"""
	embeddings = load_cached_embeddings(model_tag)
	todo = corpus_df[~corpus_df["id"].isin(embeddings)]
	if todo.empty:
		print(f"{model_tag}: {len(embeddings)} embeddings already cached, nothing to do.")
		return embeddings

	print(f"{model_tag}: embedding {len(todo)} of {len(corpus_df)} tickets ({len(embeddings)} already cached)...")
	failures: list[int] = []
	for n, row in enumerate(tqdm(list(todo.itertuples()), desc=model_tag), start=1):
		try:
			vector, _ = embed_one(model_tag, row.description)
			embeddings[row.id] = np.asarray(vector, dtype=np.float32)
		except EmbeddingFailure as exc:
			print(f"  skipped ticket id={row.id}: {exc}")
			failures.append(row.id)
		if n % checkpoint_every == 0:
			save_cached_embeddings(model_tag, embeddings)

	save_cached_embeddings(model_tag, embeddings)
	if failures:
		print(f"{model_tag}: {len(failures)} tickets failed after retries -- ids: {failures}")
	return embeddings


In [ ]:
embeddings_by_model: dict[str, dict[int, np.ndarray]] = {}
for tag in MODEL_TAGS:
	embeddings_by_model[tag] = embed_corpus(tag, corpus)


## 5. Retrieval: top-10 nearest neighbors per query, per model

Cosine similarity via normalized dot product -- the same formula production's `pgvector_similarity_search.py` uses (`1 - cosine_distance`), computed here in numpy instead of SQL.

In [ ]:
def cosine_top_k(query_vec: np.ndarray, corpus_ids: np.ndarray, corpus_matrix: np.ndarray, exclude_id: int, k: int) -> list[tuple[int, float]]:
	q = query_vec / np.linalg.norm(query_vec)
	corpus_norms = corpus_matrix / np.linalg.norm(corpus_matrix, axis=1, keepdims=True)
	scores = corpus_norms @ q
	order = np.argsort(-scores)
	results: list[tuple[int, float]] = []
	for idx in order:
		cid = int(corpus_ids[idx])
		if cid == exclude_id:
			continue
		results.append((cid, float(scores[idx])))
		if len(results) == k:
			break
	return results


def retrieve_top_k_for_model(model_tag: str, embeddings: dict[int, np.ndarray], query_ids: list[int], k: int = TOP_K) -> dict[int, list[tuple[int, float]]]:
	cache_path = RETRIEVAL_DIR / f"{model_tag.replace(':', '_')}.json"
	if cache_path.exists():
		raw = json.loads(cache_path.read_text(encoding="utf-8"))
		return {int(qid): [(int(cid), float(s)) for cid, s in pairs] for qid, pairs in raw.items()}

	corpus_ids = np.array(sorted(embeddings))
	corpus_matrix = np.stack([embeddings[i] for i in corpus_ids])
	results = {qid: cosine_top_k(embeddings[qid], corpus_ids, corpus_matrix, exclude_id=qid, k=k) for qid in query_ids}
	cache_path.write_text(json.dumps({str(qid): pairs for qid, pairs in results.items()}), encoding="utf-8")
	return results


In [ ]:
retrieval_by_model: dict[str, dict[int, list[tuple[int, float]]]] = {}
for tag in MODEL_TAGS:
	retrieval_by_model[tag] = retrieve_top_k_for_model(tag, embeddings_by_model[tag], query_ids)
	print(f"{tag}: retrieval ready for {len(retrieval_by_model[tag])} queries.")


## 6. Human relevance judgment (pooled, blinded, reusable)

Each model's top-10 candidates per query are pooled into one deduplicated queue; the evaluator judges each (query, candidate) pair exactly once and never sees which model(s) actually retrieved it -- blinding falls out of the pooling design for free. Binary relevant / not relevant, per the earlier decision.

In [ ]:
def load_judgments() -> dict[tuple[int, int], bool]:
	if not JUDGMENTS_PATH.exists():
		return {}
	judgments: dict[tuple[int, int], bool] = {}
	with JUDGMENTS_PATH.open(encoding="utf-8") as fh:
		for line in fh:
			line = line.strip()
			if not line:
				continue
			record = json.loads(line)
			judgments[(record["query_id"], record["candidate_id"])] = record["relevant"]
	return judgments


def append_judgment(query_id: int, candidate_id: int, relevant: bool) -> None:
	"""Appends immediately (JSONL, not a rewritten file) so a judging session can be interrupted at
	any point without losing prior work.
	"""
	record = {"query_id": query_id, "candidate_id": candidate_id, "relevant": relevant, "judged_at": time.time()}
	with JUDGMENTS_PATH.open("a", encoding="utf-8") as fh:
		fh.write(json.dumps(record) + "\n")


def build_pooled_pairs(retrieval_by_model: dict[str, dict[int, list[tuple[int, float]]]], query_ids: list[int]) -> list[tuple[int, int]]:
	"""Union of every model's top-K candidates per query, deduplicated. Each (query, candidate)
	pair is judged at most once and the judgment is reused for every model that retrieved it.
	"""
	pairs: list[tuple[int, int]] = []
	seen: set[tuple[int, int]] = set()
	for qid in query_ids:
		candidates: set[int] = set()
		for results in retrieval_by_model.values():
			candidates.update(cid for cid, _ in results[qid])
		for cid in sorted(candidates):
			if (qid, cid) not in seen:
				seen.add((qid, cid))
				pairs.append((qid, cid))
	return pairs


judgments = load_judgments()
pooled_pairs = build_pooled_pairs(retrieval_by_model, query_ids)
remaining_pairs = [p for p in pooled_pairs if p not in judgments]

print(f"Pooled (query, candidate) pairs across all {len(MODEL_TAGS)} models: {len(pooled_pairs)}")
print(f"Already judged: {len(judgments)} | Remaining: {len(remaining_pairs)}")


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_judge_state = {"index": 0, "pairs": remaining_pairs}


def _render_pair(query_id: int, candidate_id: int) -> str:
	q = corpus_by_id.loc[query_id]
	c = corpus_by_id.loc[candidate_id]
	resolution = c["resolution_notes"] if pd.notna(c["resolution_notes"]) else "(none recorded)"
	return (
		f"<h4>Query ticket #{query_id} -- {q['application']} / {q['category']}</h4>"
		f"<p><b>{q['title']}</b></p><p>{q['description']}</p>"
		"<hr>"
		f"<h4>Candidate ticket #{candidate_id} -- {c['application']} / {c['category']}</h4>"
		f"<p><b>{c['title']}</b></p><p>{c['description']}</p>"
		f"<p><i>Resolution notes:</i> {resolution}</p>"
	)


_output = widgets.Output()
_progress_label = widgets.Label()
_relevant_btn = widgets.Button(description="Relevant", button_style="success")
_not_relevant_btn = widgets.Button(description="Not relevant", button_style="danger")
_skip_btn = widgets.Button(description="Skip for now")


def _show_current() -> None:
	pairs = _judge_state["pairs"]
	idx = _judge_state["index"]
	_progress_label.value = f"{idx} / {len(pairs)} judged this session -- {len(pairs) - idx} remaining"
	with _output:
		clear_output(wait=True)
		if idx >= len(pairs):
			print("All pooled pairs judged. If you add another model later, re-run the pooling cell "
				"above and this cell to pick up only the newly-added pairs.")
			return
		qid, cid = pairs[idx]
		display(widgets.HTML(_render_pair(qid, cid)))


def _on_relevant(_) -> None:
	qid, cid = _judge_state["pairs"][_judge_state["index"]]
	append_judgment(qid, cid, True)
	_judge_state["index"] += 1
	_show_current()


def _on_not_relevant(_) -> None:
	qid, cid = _judge_state["pairs"][_judge_state["index"]]
	append_judgment(qid, cid, False)
	_judge_state["index"] += 1
	_show_current()


def _on_skip(_) -> None:
	pairs = _judge_state["pairs"]
	pairs.append(pairs.pop(_judge_state["index"]))
	_show_current()


_relevant_btn.on_click(_on_relevant)
_not_relevant_btn.on_click(_on_not_relevant)
_skip_btn.on_click(_on_skip)

display(_progress_label, _output, widgets.HBox([_relevant_btn, _not_relevant_btn, _skip_btn]))
_show_current()


## 7. Retrieval quality metrics

Precision@10 and MRR are computed directly; Recall@10 is *pool-relative* (relative to the union of relevant items any model found for a query), not true corpus-wide recall -- see the caveat in the docstring below.

In [ ]:
def compute_metrics(retrieval_by_model: dict[str, dict[int, list[tuple[int, float]]]], judgments: dict[tuple[int, int], bool], query_ids: list[int]) -> pd.DataFrame:
	"""Precision@K and MRR are well-defined per (model, query) regardless of pooling -- they only
	depend on judgments for that model's own retrieved items, which are always in the pool by
	construction. Recall@K is necessarily *pool-relative*: relative to the union of relevant items
	ANY model found for that query, not true corpus-wide recall (which would require exhaustively
	judging the whole corpus -- exactly what pooling exists to avoid). Queries where the pool
	contains zero relevant candidates are excluded from the recall average (undefined, not zero)
	but still contribute to precision and MRR, both of which are correctly 0 in that case.
	"""
	pool_relevant_by_query: dict[int, set[int]] = {qid: set() for qid in query_ids}
	for (qid, cid), relevant in judgments.items():
		if relevant and qid in pool_relevant_by_query:
			pool_relevant_by_query[qid].add(cid)

	rows = []
	for model_tag, results in retrieval_by_model.items():
		for qid in query_ids:
			retrieved = [cid for cid, _ in results[qid]]
			relevant_retrieved = [cid for cid in retrieved if judgments.get((qid, cid), False)]
			precision = len(relevant_retrieved) / len(retrieved) if retrieved else 0.0
			pool_relevant = pool_relevant_by_query[qid]
			recall = len(relevant_retrieved) / len(pool_relevant) if pool_relevant else float("nan")
			reciprocal_rank = 0.0
			for rank, cid in enumerate(retrieved, start=1):
				if judgments.get((qid, cid), False):
					reciprocal_rank = 1.0 / rank
					break
			rows.append({
				"model": model_tag,
				"query_id": qid,
				"application": corpus_by_id.loc[qid, "application"],
				"precision_at_k": precision,
				"recall_at_k_pool": recall,
				"reciprocal_rank": reciprocal_rank,
			})
	return pd.DataFrame(rows)


In [ ]:
metrics_df = compute_metrics(retrieval_by_model, judgments, query_ids)

summary = metrics_df.groupby("model").agg(
	precision_at_k=("precision_at_k", "mean"),
	recall_at_k_pool=("recall_at_k_pool", "mean"),
	mrr=("reciprocal_rank", "mean"),
	n_queries=("query_id", "nunique"),
	n_recall_eligible=("recall_at_k_pool", lambda s: int(s.notna().sum())),
)
print(f"Queries with zero relevant candidates found by any model (excluded from recall average): "
	f"{summary['n_queries'].iloc[0] - summary['n_recall_eligible'].iloc[0]} / {len(query_ids)}")
summary


In [ ]:
per_app_summary = metrics_df.groupby(["model", "application"]).agg(
	precision_at_k=("precision_at_k", "mean"),
	recall_at_k_pool=("recall_at_k_pool", "mean"),
	mrr=("reciprocal_rank", "mean"),
	n=("query_id", "nunique"),
)
per_app_summary


## 8. Practical performance benchmarking

Cold-start (model load) and warm single-call latency, timed via Ollama's own server-side duration, plus resident memory via `ollama ps` -- all on this exact CPU-only, no-GPU machine, which is the actual constraint we're planning around.

In [ ]:
def sample_latency_texts(corpus_df: pd.DataFrame, n: int, seed: int) -> list[str]:
	"""A fixed sample of real description texts, shared across models, so every model is timed on
	identical inputs spanning the corpus's natural length range.
	"""
	rng = np.random.default_rng(seed)
	idx = rng.choice(len(corpus_df), size=min(n, len(corpus_df)), replace=False)
	return corpus_df.iloc[idx]["description"].tolist()


def benchmark_model(model_tag: str, texts: list[str]) -> dict:
	"""Cold start: force-unloads the model first (keep_alive=0) so the measurement isn't an
	artifact of the model already being resident from the embedding-generation step above, then
	times the reload. Warm latency: repeated single-item calls after that, matching production's
	actual access pattern (GenerateSimilarityResultsHandler embeds exactly one description per
	call, never a batch) -- so single-call latency IS the realistic throughput proxy here, not an
	approximation of it.
	"""
	try:
		_client.embed(model=model_tag, input="unload probe", keep_alive=0)
	except Exception:
		pass
	time.sleep(1.0)

	_, cold_duration_ns = embed_one(model_tag, texts[0])

	durations_ms = []
	for text in texts[1:]:
		_, duration_ns = embed_one(model_tag, text)
		durations_ms.append(duration_ns / 1e6)

	running = next((m for m in _client.ps().models if m.model == _full_tag(model_tag)), None)

	return {
		"model_tag": model_tag,
		"cold_start_ms": cold_duration_ns / 1e6,
		"warm_latency_mean_ms": float(np.mean(durations_ms)),
		"warm_latency_median_ms": float(np.median(durations_ms)),
		"warm_latency_p95_ms": float(np.percentile(durations_ms, 95)),
		"approx_throughput_per_sec": 1000.0 / float(np.mean(durations_ms)),
		"resident_memory_mb": (running.size / 1e6) if running else None,
		"resident_memory_vram_mb": (running.size_vram / 1e6) if running else None,
	}


In [ ]:
latency_texts = sample_latency_texts(corpus, LATENCY_SAMPLE_SIZE, seed=RANDOM_SEED)

resource_results: dict = json.loads(RESOURCE_BENCHMARK_PATH.read_text(encoding="utf-8")) if RESOURCE_BENCHMARK_PATH.exists() else {}

for tag in MODEL_TAGS:
	if tag in resource_results:
		continue
	resource_results[tag] = benchmark_model(tag, latency_texts)
	RESOURCE_BENCHMARK_PATH.write_text(json.dumps(resource_results, indent=2), encoding="utf-8")  # checkpoint after each model

resource_df = pd.DataFrame(list(resource_results.values())).assign(
	disk_size_mb=lambda d: d["model_tag"].map(lambda t: model_info[t]["disk_size_bytes"] / 1e6),
)
resource_df


## 9. Embedding space visualization (PCA & UMAP) -- diagnostic only

Not a selection criterion by itself -- used to spot structure or failure patterns that complement the quantitative metrics above. "Nicer-looking" clusters do not imply better retrieval.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def plot_projection(projector_name: str, transform_fn) -> None:
	"""Projects the full retrievable corpus (not just the 100 queries) for more visible structure;
	sampled queries are outlined for reference.
	"""
	fig, axes = plt.subplots(1, len(MODEL_TAGS), figsize=(6 * len(MODEL_TAGS), 5))
	apps = sorted(corpus_by_id["application"].unique())
	colors = {app: plt.cm.tab10(i) for i, app in enumerate(apps)}
	for ax, tag in zip(axes, MODEL_TAGS):
		emb = embeddings_by_model[tag]
		ids = np.array(sorted(emb))
		matrix = np.stack([emb[i] for i in ids])
		coords = transform_fn(matrix)
		app_labels = corpus_by_id.loc[ids, "application"]
		for app in apps:
			mask = (app_labels == app).to_numpy()
			ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.5, color=colors[app], label=app)
		is_query = np.isin(ids, query_ids)
		ax.scatter(
			coords[is_query, 0], coords[is_query, 1],
			s=40, facecolors="none", edgecolors="black", linewidths=0.8, label="query",
		)
		ax.set_title(f"{projector_name}: {tag}")
		ax.legend(fontsize=7, loc="best")
	plt.tight_layout()
	plt.show()


plot_projection("PCA", lambda m: PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(m))


In [ ]:
import umap

plot_projection(
	"UMAP",
	lambda m: umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=15, min_dist=0.1).fit_transform(m),
)


## 10. Comparative analysis: quality vs. cost

Side by side, no weighted composite score -- the trade-off is argued in the final write-up (section 12), not collapsed into one number.

In [ ]:
comparison = summary.join(
	resource_df.set_index("model_tag")[[
		"cold_start_ms", "warm_latency_mean_ms", "warm_latency_p95_ms",
		"approx_throughput_per_sec", "resident_memory_mb", "disk_size_mb",
	]]
)
comparison


In [ ]:
quality_cols = [
	("precision_at_k", "Precision@10"),
	("recall_at_k_pool", "Recall@10 (pool-relative)"),
	("mrr", "MRR"),
]
fig, axes = plt.subplots(1, len(quality_cols), figsize=(5 * len(quality_cols), 4))
for ax, (col, title) in zip(axes, quality_cols):
	comparison[col].plot(kind="bar", ax=ax, color="#4C72B0")
	ax.set_title(title)
	ax.set_xlabel("")
plt.tight_layout()
plt.show()

cost_cols = [
	("warm_latency_mean_ms", "Warm latency (ms/call)"),
	("resident_memory_mb", "Resident memory (MB)"),
	("disk_size_mb", "Disk size (MB)"),
]
fig, axes = plt.subplots(1, len(cost_cols), figsize=(5 * len(cost_cols), 4))
for ax, (col, title) in zip(axes, cost_cols):
	comparison[col].plot(kind="bar", ax=ax, color="#DD8452")
	ax.set_title(title)
	ax.set_xlabel("")
plt.tight_layout()
plt.show()


## 11. Failure cases & model disagreement

Queries where models diverge most, for qualitative inspection -- not scoring.

In [ ]:
def show_disagreement_cases(n: int = 5) -> None:
	"""Surfaces queries where models disagree most on Precision@10 -- the largest max-min spread --
	for qualitative reading. Useful for spotting systematic failure modes (e.g. one model
	consistently missing short FCI-style descriptions).
	"""
	spread = (
		metrics_df.pivot(index="query_id", columns="model", values="precision_at_k")
		.assign(spread=lambda d: d.max(axis=1) - d.min(axis=1))
		.sort_values("spread", ascending=False)
	)
	for qid in spread.head(n).index:
		q = corpus_by_id.loc[qid]
		print(f"=== Query #{qid} [{q['application']}] precision spread={spread.loc[qid, 'spread']:.2f} ===")
		print(f"{q['title']}: {q['description'][:200]}")
		for tag in MODEL_TAGS:
			top3 = retrieval_by_model[tag][qid][:3]
			marks = ["+" if judgments.get((qid, cid), False) else "-" for cid, _ in top3]
			line = ", ".join(f"#{cid}({score:.2f}){mark}" for (cid, score), mark in zip(top3, marks))
			print(f"  {tag}: {line}")
		print()


show_disagreement_cases()


## 12. Decision

Fill in after running the full evaluation (embeddings + judgments complete for all three models).

**Quality ranking** (Precision@10 / Recall@10-pool / MRR, overall and per application):
_..._

**Cost ranking** (latency, resident memory, disk size) on this machine:
_..._

**Is the quality gap large enough to justify the more expensive model?**
_..._

**Recommendation:**
_..._

**Known limitations of this evaluation** (carry these into the decision -- don't treat the numbers
as more precise than they are):

- Small corpus (799 tickets total) and especially thin signal for AERO (54 tickets, ~7 queries).
- Single annotator, binary relevance -- no inter-annotator agreement measure.
- Recall@10 is pool-relative, not true corpus recall.
- Resource numbers are specific to this CPU-only dev machine and Ollama's local scheduling; they
  may not transfer exactly to wherever this ends up running in production.
- Description-only representation, per the fixed scope of this experiment -- title/metadata/
  category are not considered here even though they exist on every ticket.
